In [ ]:
import datetime
import numpy as np
from numpy.polynomial import Chebyshev
import pandas as pd
import scipy
import matplotlib.pyplot as plt


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(precision=2, suppress=True)

In [ ]:
filename = r'TSLA_20240918_161642_H.csv'
df = pd.read_csv(filename, index_col=0, parse_dates=['date'])

In [ ]:
df.info()

In [ ]:
# df['t'] = df.index.to_series().diff().dt.total_seconds().div(60).fillna(0)
# df['t'] = df.groupby(df.index.date)['t'].cumsum()


In [ ]:
ann_fact = np.sqrt(252 * 1440)
df['close - average'] = df['close'] - df['average']
df['close - open'] = df['close'] - df['open']
df['t_base'] = df.groupby(df['date'].dt.date)['date'].transform('min')
df['price_base'] = df.groupby(df['date'].dt.date)['open'].transform('first')
df['chg'] = df['close'] - df['price_base'] # period price change
df['logret'] = np.log(df['close'] / df['price_base']) # period log return, close / base (first open)
df['dt'] = (df['date'] - df['t_base']).dt.total_seconds().div(60).add(1)
df['sqrt_dt'] = df['dt'].apply(lambda x: x ** 0.5)
df['logret/sqrt_dt'] = ann_fact * df['logret'] / df['sqrt_dt'] # rate of return, per sqrt(dt), annualized

In [ ]:
df

In [ ]:
#open_close_list = [(row['open'], row['close']) for _, row in df.iloc[-10:].iterrows()]
np.array([(row['open'], row['close']) for _, row in df.iloc[-10:].iterrows()]).flatten()
#open_close_list

In [ ]:
idx = slice(265,280)
y = df['close'].iloc[idx].to_numpy()
y_high = df['high'].iloc[idx].to_numpy()
y_low = df['low'].iloc[idx].to_numpy()
y_open = df['open'].iloc[idx].to_numpy()
x = df['dt'].iloc[idx].to_numpy()

In [ ]:
# ak_fun = scipy.interpolate.Akima1DInterpolator(x, y, method='makima', extrapolate=False)
ak_fun = scipy.interpolate.PchipInterpolator(x, y, extrapolate=False)


In [ ]:
ak_fun

In [ ]:
def objective(coeffs):
    cheb = Chebyshev(coeffs, domain=cheb_poly[0].domain)
    return np.sum((cheb(x) - y) ** 2)

def constraint(coeffs):
    cheb = Chebyshev(coeffs, domain=cheb_poly[0].domain)
    fitted = cheb(x)
    high_constraint = y_high - fitted
    low_constraint = fitted - y_low
    return np.concatenate((high_constraint, low_constraint))

initial_guess = Chebyshev.fit(x, y, degree).coef # fit without constraints
print(initial_guess)

cons = {'type': 'ineq', 'fun': constraint}
result = minimize(objective, initial_guess, method='SLSQP', constraints=cons)

In [ ]:
result

In [ ]:
objective( cheb_poly[0].coef )

In [ ]:
print(cheb_poly[0])

In [ ]:

# Generate a range of x values for plotting the polynomial
x_plot = np.linspace(x.min(), x.max(), 500)
# Evaluate the Chebyshev polynomial at the x values
y_plot = cheb_poly[0](x_plot)

# Plot the original data points and the fitted Chebyshev polynomial
plt.figure(figsize=(10, 6))
plt.plot(x, y, 'o', label='Original data')
plt.plot(x_plot, y_plot, '-', label='fitted curve')
plt.xlabel('dt')
plt.ylabel('close')
plt.legend()
# plt.title('Chebyshev Polynomial Fit')
plt.show()

In [ ]:

# Generate a range of x values for plotting the polynomial
x_plot = np.linspace(x.min(), x.max(), 500)
# Evaluate the Chebyshev polynomial at the x values
y_plot = Chebyshev(result.x, domain=cheb_poly[0].domain)(x_plot) # cheb_poly[0](x_plot)

# Plot the original data points and the fitted Chebyshev polynomial
plt.figure(figsize=(10, 6))
plt.plot(x, y, 'o', label='close')
plt.plot(x, y_high, '^', label='high')
plt.plot(x, y_low, 'v', label='low')
plt.plot(x, y_open, 'o', label='open')
plt.plot(x_plot, y_plot, '-', label='fitted curve')
plt.xlabel('dt')
plt.ylabel('close')
plt.legend()
# plt.title('Chebyshev Polynomial Fit')
plt.show()

In [ ]:
time_of_day = df['date'].iloc[idx].dt.time.to_numpy()
print(time_of_day)

In [ ]:
plt.figure(figsize=(10, 6))

x_plot = np.linspace(x.min(), x.max(), 500)
y_plot = ak_fun(x_plot)
# x_dates = [df['date'].iloc[idx[0]], df['date'].iloc[idx[-1]]]
# Calculate the derivative
der1 = ak_fun.derivative()
# Evaluate the derivative at the points in x
slopes = der1(x)

# # Plot y as a line
# plt.plot(x, y, label='y', color='blue')

# Plot y_low and y_high as lines
plt.plot(x, y_low, 'v', label='y_low', color='red')
plt.plot(x, y_high, '^',label='y_high', color='green')

plt.plot(x, y, 'x', label='close')
# plt.plot(x, y_high, '^', label='high')
# plt.plot(x, y_low, 'v', label='low')
# plt.plot(x, y_open, 'o', label='open')
plt.plot(x_plot, y_plot, '-', label='fitted curve')
# Display the slopes
for i, txt in enumerate(slopes):
    plt.annotate(f'${txt:.2f}$', (x[i], y[i]), textcoords="offset points", xytext=(0,30), ha='center', fontsize=8, usetex=True)

plt.xticks(ticks=x, labels=[t.strftime('%H:%M') for t in time_of_day], rotation=45)
# plt.xlabel('dt')

# Shade the region between y_low and y_high
plt.fill_between(x, y_low, y_high, color='gray', alpha=0.2)

# Add labels and legend
# plt.xlabel('x')
# plt.ylabel('Values')
plt.legend()
# plt.title('Plot of y, y_low, and y_high with Shaded Region')
plt.grid(True, color='#d7d7d7', linestyle='-', linewidth=0.5)
plt.show()

In [ ]:
# fig = plt.figure()
# ax = fig.add_subplot()

cols = ['sqrt_dt', 'logret', 'logret/sqrt_dt']
df[cols].plot(x='sqrt_dt', subplots=True, sharex=True)
# cols = ['sqrt_dt', 'logret/sqrt_dt']
# df[cols].plot(x='sqrt_dt', subplots=True)

In [ ]:
# BEGIN: Compute rate of return
first_open = df.iloc[0]['open']
df['rate of return'] = (df['close'] - first_open) / ((df.index - df.index[0]).total_seconds() / 3600)
df
# END: Compute rate of return

In [ ]:
df['close rate of return'] = (df['close'] - df.iloc[0]['open']) 
df

In [ ]:
df5 = df.resample('5min').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'barCount': 'sum'})   

In [ ]:
df5['pnl_5min'] = (df5['close'] - df5['open']) / 5.0 * 60

In [ ]:
df5['pnl_5min_pct'] = (df5['close'] - df5['open']) / df5['open'] / 5.0 * 60 * 24 * 252

In [ ]:
df1h = df.resample('1h').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'barCount': 'sum'})

In [ ]:
df1h['pnl_1h'] = (df1h['close'] - df1h['open']) 
df1h['pnl_1h_pct'] = (df1h['close'] - df1h['open']) / df1h['open']

In [ ]:
df1h

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt

# Sample stock price data (time, open, high, low, close)
data = np.array([
    [0, 100, 105, 98, 103],
    [1, 103, 107, 101, 105],
    [2, 105, 108, 103, 106],
    [3, 106, 110, 104, 108],
    [4, 108, 112, 107, 111],
])

# Separate the data
time = data[:, 0]
open_prices = data[:, 1]
high_prices = data[:, 2]
low_prices = data[:, 3]
close_prices = data[:, 4]

# Create a more detailed time array for smoother curve
t_smooth = np.linspace(time.min(), time.max(), 1000)

# Fit cubic spline to close prices
cs_close = CubicSpline(time, close_prices)

# Fit cubic splines to high and low prices
cs_high = CubicSpline(time, high_prices)
cs_low = CubicSpline(time, low_prices)

# Calculate spline values
spline_close = cs_close(t_smooth)
spline_high = cs_high(t_smooth)
spline_low = cs_low(t_smooth)

# Constrain the close spline to stay within high/low bounds
spline_constrained = np.minimum(np.maximum(spline_close, spline_low), spline_high)

# Plotting
plt.figure(figsize=(12, 6))

# Plot the original candlestick-like data
for i in range(len(time)):
    plt.plot([time[i], time[i]], [low_prices[i], high_prices[i]], 'k-')
    plt.plot([time[i]-0.1, time[i]+0.1], [open_prices[i], open_prices[i]], 'b-')
    plt.plot([time[i]-0.1, time[i]+0.1], [close_prices[i], close_prices[i]], 'r-')

# Plot the constrained spline
plt.plot(t_smooth, spline_constrained, 'g-', label='Constrained Spline')

# Plot high and low splines
plt.plot(t_smooth, spline_high, 'r--', alpha=0.5, label='High Spline')
plt.plot(t_smooth, spline_low, 'b--', alpha=0.5, label='Low Spline')

plt.title('Stock Price Spline Fit')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
from scipy.optimize import minimize

def chebyshev_poly(x, degree):
    T = [np.ones_like(x), x]
    for i in range(2, degree + 1):
        T.append(2 * x * T[i-1] - T[i-2])
    return np.array(T)

def fit_constrained_chebyshev(x, y_open, y_high, y_low, y_close, degree):
    T = chebyshev_poly(x, degree)
    
    def objective(coeffs):
        y_pred = T.T @ coeffs
        return np.mean((y_pred - y_close)**2)
    
    def constraint(coeffs):
        y_pred = T.T @ coeffs
        return np.min(y_high - y_pred) * np.min(y_pred - y_low)
    
    cons = {'type': 'ineq', 'fun': constraint}
    res = minimize(objective, np.zeros(degree + 1), method='SLSQP', constraints=cons)
    
    return res.x

# Example usage
x = np.linspace(-1, 1, len(data))  # Normalize time to [-1, 1]
y_open = data[:, 1]
y_high = data[:, 2]
y_low = data[:, 3]
y_close = data[:, 4]

degree = 5  # Choose an appropriate degree
coeffs = fit_constrained_chebyshev(x, y_open, y_high, y_low, y_close, degree)

# Generate fitted curve
x_smooth = np.linspace(-1, 1, 1000)
T_smooth = chebyshev_poly(x_smooth, degree)
y_fit = T_smooth.T @ coeffs

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(x, y_close, 'ko', label='Close')
plt.plot(x, y_high, 'r^', label='High')
plt.plot(x, y_low, 'gv', label='Low')
plt.plot(x_smooth, y_fit, 'b-', label='Fitted Curve')
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import Chebyshev

# Sample stock price data (time, open, high, low, close)
data = np.array([
    [0, 100, 105, 98, 103],
    [1, 103, 107, 101, 105],
    [2, 105, 108, 103, 106],
    [3, 106, 110, 104, 108],
    [4, 108, 112, 107, 111],
])

# Separate the data
time = data[:, 0]
open_prices = data[:, 1]
high_prices = data[:, 2]
low_prices = data[:, 3]
close_prices = data[:, 4]

# Normalize time to [-1, 1] for Chebyshev polynomials
normalized_time = 2 * (time - time.min()) / (time.max() - time.min()) - 1

# Fit Chebyshev polynomials
degree = 3  # Adjust this value as needed
cheb_close = Chebyshev.fit(normalized_time, close_prices, degree)
cheb_high = Chebyshev.fit(normalized_time, high_prices, degree)
cheb_low = Chebyshev.fit(normalized_time, low_prices, degree)

# Create a more detailed time array for smoother curve
t_smooth = np.linspace(-1, 1, 1000)

# Calculate polynomial values
fitted_close = cheb_close(t_smooth)
fitted_high = cheb_high(t_smooth)
fitted_low = cheb_low(t_smooth)

# Constrain the close fit to stay within high/low bounds
fitted_constrained = np.minimum(np.maximum(fitted_close, fitted_low), fitted_high)

# Convert t_smooth back to original time scale for plotting
t_smooth_original = (t_smooth + 1) * (time.max() - time.min()) / 2 + time.min()

# Plotting
plt.figure(figsize=(12, 6))

# Plot the original candlestick-like data
for i in range(len(time)):
    plt.plot([time[i], time[i]], [low_prices[i], high_prices[i]], 'k-')
    plt.plot([time[i]-0.1, time[i]+0.1], [open_prices[i], open_prices[i]], 'b-')
    plt.plot([time[i]-0.1, time[i]+0.1], [close_prices[i], close_prices[i]], 'r-')

# Plot the constrained fit
plt.plot(t_smooth_original, fitted_constrained, 'g-', label='Constrained Fit')

# Plot high and low fits
plt.plot(t_smooth_original, fitted_high, 'r--', alpha=0.5, label='High Fit')
plt.plot(t_smooth_original, fitted_low, 'b--', alpha=0.5, label='Low Fit')

plt.title('Stock Price Chebyshev Polynomial Fit')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()
